In [0]:
# ============================================
# 01. IMPORTACIONES
# AL no tener información bancaria real, generamos data aleatoria 
# 
# ============================================

import random
import csv
import json

from datetime import datetime, date, timedelta
from decimal import Decimal


# ============================================
# 02. CONFIGURACIÓN DEL PROYECTO
# ============================================

PROJECT_NAME = "Banking Analytics Platform"

BASE_PATH = "/Volumes/banco/banking_analytics/banking_files"

RAW_PATH = f"{BASE_PATH}/raw"

# Semilla para reproducibilidad
RANDOM_SEED = 42
random.seed(RANDOM_SEED)


# ============================================
# 03. CANTIDAD DE REGISTROS
# ============================================

N_CUSTOMERS = 10_000
N_ACCOUNTS = 15_000
N_TRANSACTIONS = 100_000
N_BRANCHES = 50


# ============================================
# 04. VALIDACIÓN DEL ALMACENAMIENTO
# ============================================

print(f"Proyecto: {PROJECT_NAME}")
print(f"Ruta de almacenamiento: {BASE_PATH}")

display(dbutils.fs.ls(BASE_PATH))

In [0]:
# ============================================
# GENERACIÓN DE AGENCIAS
# ============================================

cities = [
    ("Lima", "Costa"),
    ("Arequipa", "Sierra"),
    ("Trujillo", "Costa"),
    ("Chiclayo", "Costa"),
    ("Piura", "Costa"),
    ("Cusco", "Sierra"),
    ("Huancayo", "Sierra"),
    ("Ayacucho", "Sierra"),
    ("Iquitos", "Selva"),
    ("Pucallpa", "Selva"),
    ("Tarapoto", "Selva"),
    ("Puno", "Sierra"),
    ("Tacna", "Costa"),
    ("Cajamarca", "Sierra"),
    ("Huaraz", "Sierra")
]

branches = []

for i in range(1, N_BRANCHES + 1):

    city, region = random.choice(cities)

    branch = {
        "branch_id": f"BR{i:03d}",
        "branch_name": f"Agencia {city} {i:02d}",
        "city": city,
        "region": region,
        "country": "Peru",
        "opening_date": (
            date(2000, 1, 1) +
            timedelta(days=random.randint(0, 9000))
        ).isoformat(),
        "is_active": random.choice([True, True, True, False])
    }

    branches.append(branch)


print(f"Total de agencias generadas: {len(branches)}")

print("Primeros 5 registros:")

for branch in branches[:5]:
    print(branch)

In [0]:
# ============================================
# EXPORTACIÓN DE AGENCIAS A CSV
# ============================================

branches_path = f"{RAW_PATH}/branches/branches.csv"

# manager context para la creación de CSV
with open(branches_path, mode="w", newline="", encoding="utf-8") as file:

    writer = csv.DictWriter(
        file,
        fieldnames=branches[0].keys()
    )

    writer.writeheader()
    writer.writerows(branches)


print(f"Archivo generado: {branches_path}")

In [0]:
# Validación de la creación del archivo en el RAW/branches
display(dbutils.fs.ls(f"{RAW_PATH}/branches"))

In [0]:
# ============================================
# GENERACIÓN DE CUSTOMERS
# ============================================

first_names = [
    "Carlos", "Luis", "Jorge", "Miguel", "Andres",
    "Pedro", "Jose", "Diego", "Fernando", "Ricardo",
    "Maria", "Ana", "Carmen", "Rosa", "Lucia",
    "Sofia", "Valeria", "Daniela", "Patricia", "Elena"
]

last_names = [
    "Garcia", "Rodriguez", "Perez", "Lopez", "Torres",
    "Flores", "Sanchez", "Ramirez", "Castillo", "Vargas",
    "Rojas", "Mendoza", "Gutierrez", "Chavez", "Diaz"
]

segments = ["STANDARD", "GOLD", "PLATINUM", "PREMIUM"]

countries = ["Peru", "Peru", "Peru", "Chile", "Colombia"]

customers = []

start_birth_date = date(1940, 1, 1)
end_birth_date = date(2005, 12, 31)

start_registration = date(2010, 1, 1)
end_registration = date(2026, 10, 31)

for i in range(1, N_CUSTOMERS + 1):

    first_name = random.choice(first_names)
    last_name = random.choice(last_names)

    customer_name = f"{first_name} {last_name}"

    birth_date = (
        start_birth_date +
        timedelta(days=random.randint(
            0,
            (end_birth_date - start_birth_date).days
        ))
    )

    registration_date = (
        start_registration +
        timedelta(days=random.randint(
            0,
            (end_registration - start_registration).days
        ))
    )

    customer = {
        "customer_id": f"CUST{i:06d}",
        "customer_name": customer_name,
        "birth_date": birth_date.isoformat(),
        "country": random.choice(countries),
        "registration_date": registration_date.isoformat(),
        "email": f"{first_name.lower()}.{last_name.lower()}{i}@email.com",
        "customer_segment": random.choice(segments)
    }

    customers.append(customer)


print(f"Clientes generados: {len(customers)}")

for customer in customers[:5]:
    print(customer)

In [0]:
# ============================================
# INYECCIÓN DE ERRORES DE CALIDAD
# ============================================

# 1. Valores nulos
for i in range(100):
    customers[i]["customer_name"] = None

for i in range(100, 150):
    customers[i]["email"] = None


# 2. Fechas de nacimiento inconsistentes
for i in range(150, 160):
    customers[i]["birth_date"] = "2035-05-20"


# 3. Correos con formato incorrecto
for i in range(160, 180):
    customers[i]["email"] = "correo_invalido"


# 4. Duplicados intencionales
customers.extend([
    customers[200].copy(),
    customers[300].copy(),
    customers[400].copy()
])


print(f"Total de registros después de introducir errores: {len(customers)}")

In [0]:
# ============================================
# EXPORTACIÓN DE CUSTOMERS A CSV
# ============================================

customers_path = f"{RAW_PATH}/customers/customers.csv"

with open( customers_path,mode="w",newline="",encoding="utf-8" ) as file:

    writer = csv.DictWriter(
        file,
        fieldnames=customers[0].keys()
    )

    writer.writeheader()
    writer.writerows(customers)


print(f"Archivo generado: {customers_path}")

In [0]:
# Verificar la carga de Customers
display(dbutils.fs.ls(f"{RAW_PATH}/customers"))

In [0]:
# consultar una muestra
display(spark.read.option("header", True).csv(customers_path).limit(10))

In [0]:
# ============================================
# GENERACIÓN DE ACCOUNTS
# ============================================

account_types = [
    "SAVINGS",
    "CHECKING",
    "FIXED_TERM"
]

account_statuses = [
    "ACTIVE",
    "ACTIVE",
    "ACTIVE",
    "INACTIVE",
    "BLOCKED"
]

currencies = ["PEN", "USD"]

# Obtener identificadores de clientes y agencias
customer_ids = [
    customer["customer_id"]
    for customer in customers[:N_CUSTOMERS]
]

branch_ids = [
    branch["branch_id"]
    for branch in branches
]

accounts = []

start_opening_date = date(2010, 1, 1)
end_opening_date = date(2026, 10, 31)

for i in range(1, N_ACCOUNTS + 1):

    opening_date = (
        start_opening_date +
        timedelta(days=random.randint(
            0,
            (end_opening_date - start_opening_date).days
        ))
    )

    account = {
        "account_id": f"ACC{i:06d}",
        "customer_id": random.choice(customer_ids),
        "account_type": random.choice(account_types),
        "balance": round(random.uniform(100, 50000), 2),
        "currency": random.choice(currencies),
        "opening_date": opening_date.isoformat(),
        "branch_id": random.choice(branch_ids),
        "account_status": random.choice(account_statuses)
    }

    accounts.append(account)


print(f"Total de cuentas generadas: {len(accounts)}")

for account in accounts[:5]:
    print(account)

In [0]:
# ============================================
# INYECCIÓN DE ERRORES
# ============================================

# 1. Valores nulos en customer_id
for i in range(100):
    accounts[i]["customer_id"] = None


# 2. Referencias a clientes inexistentes
for i in range(100, 120):
    accounts[i]["customer_id"] = "CUST999999"


# 3. Saldos inválidos
for i in range(120, 130):
    accounts[i]["balance"] = -500.00


# 4. Tipos de cuenta inválidos
for i in range(130, 140):
    accounts[i]["account_type"] = "UNKNOWN"


# 5. Duplicados intencionales
accounts.extend([
    accounts[200].copy(),
    accounts[300].copy(),
    accounts[400].copy()
])


print(f"Total de registros: {len(accounts)}")

In [0]:
# ============================================
# EXPORTACIÓN DE ACCOUNTS
# ============================================

accounts_path = f"{RAW_PATH}/accounts/accounts.csv"

with open(accounts_path,mode="w",newline="",encoding="utf-8") as file:

    writer = csv.DictWriter(
        file,
        fieldnames=accounts[0].keys()
    )

    writer.writeheader()
    writer.writerows(accounts)


print(f"Archivo generado: {accounts_path}")

In [0]:
# Verificar la carga correcta
display(dbutils.fs.ls(f"{RAW_PATH}/accounts"))

In [0]:
# Leer una muestra de accounts
df_accounts = (spark.read.option("header", True).option("inferSchema", True).csv(accounts_path))

display(df_accounts.limit(10))

In [0]:
# Validacion de totales
print("Total de cuentas:", df_accounts.count())
print("Clientes distintos:", df_accounts.select("customer_id").distinct().count())
print("Agencias distintas:", df_accounts.select("branch_id").distinct().count())

In [0]:
# ============================================
# GENERACIÓN DE TRANSACTIONS
# ============================================

transaction_types = [
    "DEPOSIT",
    "WITHDRAWAL",
    "TRANSFER",
    "PAYMENT"
]

transaction_statuses = [
    "COMPLETED",
    "COMPLETED",
    "COMPLETED",
    "PENDING",
    "REJECTED"
]

channels = [
    "MOBILE_APP",
    "WEB",
    "ATM",
    "BRANCH",
    "POS"
]

descriptions = {
    "DEPOSIT": "Cash deposit",
    "WITHDRAWAL": "Cash withdrawal",
    "TRANSFER": "Account transfer",
    "PAYMENT": "Service payment"
}

# Obtener identificadores de cuentas existentes
account_ids = [
    account["account_id"]
    for account in accounts[:N_ACCOUNTS]
]

transactions = []

start_transaction_date = datetime(2025, 1, 1)
end_transaction_date = datetime(2026, 10, 31, 23, 59, 59)

total_seconds = int(
    (end_transaction_date - start_transaction_date).total_seconds()
)

for i in range(1, N_TRANSACTIONS + 1):

    transaction_type = random.choice(transaction_types)

    transaction_date = (
        start_transaction_date +
        timedelta(seconds=random.randint(0, total_seconds))
    )

    transaction = {
        "transaction_id": f"TRX{i:08d}",
        "account_id": random.choice(account_ids),
        "transaction_date": transaction_date.isoformat(),
        "transaction_type": transaction_type,
        "amount": round(random.uniform(5, 10000), 2),
        "currency": random.choice(currencies),
        "status": random.choice(transaction_statuses),
        "channel": random.choice(channels),
        "description": descriptions[transaction_type]
    }

    transactions.append(transaction)


print(f"Total de transacciones generadas: {len(transactions)}")

for transaction in transactions[:5]:
    print(transaction)

In [0]:
# ============================================
# INYECCIÓN DE ERRORES
# ============================================

# 1. account_id nulo
for i in range(100):
    transactions[i]["account_id"] = None


# 2. Referencias a cuentas inexistentes
for i in range(100, 120):
    transactions[i]["account_id"] = "ACC999999"


# 3. Importes negativos
for i in range(120, 130):
    transactions[i]["amount"] = -250.00


# 4. Tipos de transacciones inválidos
for i in range(130, 140):
    transactions[i]["transaction_type"] = "UNKNOWN"


# 5. Fechas inválidas
for i in range(140, 150):
    transactions[i]["transaction_date"] = "INVALID_DATE"


# 6. Duplicados intencionales
transactions.extend([
    transactions[200].copy(),
    transactions[300].copy(),
    transactions[400].copy()
])


print(f"Total de registros después de errores: {len(transactions)}")

In [0]:
# ============================================
# EXPORTACIÓN DE TRANSACTIONS A JSONL
# ============================================

transactions_path = f"{RAW_PATH}/transactions/transactions.jsonl"

with open(transactions_path,mode="w",encoding="utf-8") as file:

    for transaction in transactions:
        file.write(json.dumps(transaction) + "\n")


print(f"Archivo generado: {transactions_path}")

In [0]:
#Validar los datos
df_transactions = (spark.read.option("multiLine", False).json(transactions_path))

display(df_transactions.limit(10))

In [0]:
print("Total de transacciones:", df_transactions.count())

print("Cuentas distintas:",df_transactions.select("account_id").distinct().count())

In [0]:
df_transactions.printSchema()